# Cherchez les erreurs

<img src="https://p9.storage.canalblog.com/94/95/695862/79581676_o.png" alt="drawing" width="500"/>

Ce notebook part d'un **code open-access issu d'un article publié dans une revue reconnue à comité de lecture** et sert de support de cours pour :

1. **Identifier des erreurs méthodologiques** (fuite de données, mauvaise CV, mauvaise métrique, etc.).
2. **Observer l'impact** de ces erreurs sur les performances affichées.
3. **Proposer et implémenter des corrections** (pipelines, séparation train/test, recherche d'hyperparamètres correcte, etc.).

## Mode d'emploi
- Les sections **« CODE ORIGINAL »** reproduisent (volontairement) des pratiques discutables.
- Les sections **« CORRECTION »** proposent des variantes plus correctes.


---

### 📌 A noter :
- Objectif : **classification binaire** (pass/fail)
- La base de données contient des données issues de 56 plans IMRT Halcyon (sein +/- boot) (**318 champs**)
- Les variables sont **7 métriques de complexité** calculées à partir des RT Plan.  
- Les cibles sont les **résultats PSQA** mesurés avec l'imageur Portal
- **GPR 2%/2.5mm, TL à 95%**, pas de précision sur la normalisation en dose - local ou global - ni sur le seuil d'analyse : choix de ce critère car déséquilibre de classe important sur les autres critères.
- Répartition pass/fail : **175/143 (55%/45%)**

---

<font size = 6>⚠️</font>
**Sélection des métriques de complexité**

- Parmi les 7 métriques initiales, seules 3 sont conservées.
- La sélection est réalisée sur **toute la base de données** : cela introduit une **fuite de données**, car l’information des futurs exemples de test influence déjà le choix des variables.
- En pratique, la sélection de variables doit être faite **uniquement à partir des données d’entraînement**, et idéalement **intégrée dans un pipeline** via une validation croisée.

- La sélection est basée sur un **test de Student** (comparaison pass vs fail, variable par variable).  
  Ce choix est utile pour une **analyse exploratoire**, mais limité pour la sélection de variables en ML :
  - il ne teste qu’une **différence de moyenne**,
  - il reste **univarié** (ignore les interactions entre variables),
  - il repose sur des hypothèses (normalité) non documentées dans l'article.

- Une analyse de **corrélation** entre variables aurait aussi été utile (en particulier pour l’interprétation et pour certains modèles utikisés ici comme les régressions ou KNN), même si ce n’est pas une obligation absolue pour tous les algorithmes (ex: arbres / forêts).

- D’autres approches de présélection peuvent être envisagées selon la nature des variables (information mutuelle, tests non paramétriques, etc.), mais elles doivent elles aussi être appliquées **sans fuite de données**.

<font size = 6>⚠️</font>  
Vu le faible nombre de métriques, il aurait aussi été pertinent de comparer le modèle ML à des approches plus simples (ex. règle à seuil sur une ou plusieurs métriques).  
C’est une bonne pratique pour quantifier l’apport réel du modèle en performance **et** en complexité.

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import warnings

from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    confusion_matrix, roc_auc_score, roc_curve, mean_squared_error,
    accuracy_score, precision_score, f1_score, mean_absolute_error, median_absolute_error
)

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.ensemble import RandomForestClassifier

import warnings  # Pour éviter les warnings inutiles
warnings.simplefilter(action='ignore') 

## Import des données

In [ ]:
# ============================================================================
# IMPORT DES DONNÉES
# ============================================================================
df           = pd.read_excel('content/Data_erreur.xlsx')
list_to_drop = ['ID Patient', 'Nom du Case', 'G_2.5/2.5', 'G_3/3', 'G_2/3', 'G_2/2.5', 'G_3/2', 'ID faisceau ']
df_drop      = df.drop(list_to_drop, axis =1)
df_drop

In [ ]:
Y_df = df_drop['Class_2/2.5']
#There is not interest to keep other metrics than SAS10 BA BI as statistical anaylysis have shown
X_df = df_drop[['SAS10', 'BA', 'BI']]

## Préparation des données

### 🚫 CODE ORIGINAL

<font size = 6> 🚫 </font> 

In [ ]:
## PUBLI ORIGINALE
# ============================================================================
# PRÉPARATION DES DONNÉES
# ============================================================================
X_np   = np.array(X_df)
sc     = StandardScaler()
X_norm = sc.fit_transform(X_np)
X      = X_norm

X = pd.DataFrame(X, columns=['SAS10', 'BA', 'BI'])
Y = pd.DataFrame(Y_df, columns=['Class_2/2.5'])

# ============================================================================
# SPLIT TRAIN/TEST
# ============================================================================
X_train, X_test, y_train, y_test = train_test_split(X,Y['Class_2/2.5'], test_size = 0.20, random_state = 10)

### ✅ CORRECTION

Dans le code original :  
❌ Les données sont standardisées **avant** la séparation train/test -> **fuite de données** (les paramètres de standardisation utilisent indirectement l’information de tout le jeu de données).  
❌ Le split n’est pas **stratifié** -> la proportion pass/fail peut varier entre train et test et rendre l’évaluation moins stable.  
✅ Bon réflexe : faire le split d’abord (stratifié), puis intégrer la standardisation dans un **Pipeline**.

<font size = 6> ✅ : </font>  

In [ ]:
## CORRECTION
# ============================================================================
# PRÉPARATION DES DONNÉES
# ============================================================================
# Analyse du déséquilibre de classe
print("=" * 60)
print("ANALYSE DU DÉSÉQUILIBRE DE CLASSE")
print("=" * 60)
unique, counts = np.unique(Y_df, return_counts=True)
total = len(Y_df)
ratio = counts.max() / counts.min()

for classe, count in zip(unique, counts):
    print(f"Classe {classe}: {count:4d} ({count/total*100:5.1f}%)")
print("-" * 60)
print(f"Ratio: {ratio:.2f}:1")
print(f"Classe minoritaire: {counts.min()/total*100:.1f}%")
print("=" * 60)

# ============================================================================
# SPLIT TRAIN/TEST
# ============================================================================
X_train_corr, X_test_corr, y_train_corr, y_test_corr = train_test_split(
    X_df, Y_df, 
    test_size=0.2, 
    random_state=42,
    stratify=Y  # stratifier par classe
)

print(f"\nTaille train: {len(X_train)} | Taille test: {len(X_test_corr)}")
print(f"Distribution train: {np.bincount(y_train_corr)}")
print(f"Distribution test:  {np.bincount(y_test_corr)}")


✅ Toujours commencer par explorer les données :  
- distribution des classes (pass/fail),  
- valeurs manquantes,  
- outliers,  
- ordres de grandeur des variables.

En classification, connaître le **déséquilibre de classe** est essentiel pour choisir des métriques pertinentes (accuracy seule, AUC, sensibilité/spécificité, balanced accuracy, etc.).

## Recherche des hyperparamètres

### 🚫 CODE ORIGINAL

<font size = 6> 🚫 </font>  

In [ ]:
## PUBLI ORIGINALE
# ============================================================================
# CONFIGURATION DES Modèles ET GRID SEARCH
# ============================================================================

# Modèle LinearDiscriminant 
LinearDiscriminant_parameters = {'solver' : ['svd', 'lsqr', 'eigen'], 
                                 'store_covariance' : [True, False],
                                 'tol' : [0.0001,0.0002,0.0003]}

LinearDiscriminant_GridSearchCV = GridSearchCV(estimator = LinearDiscriminantAnalysis(), param_grid = LinearDiscriminant_parameters, cv = 5, n_jobs=-1)
LinearDiscriminant_GridSearchCV.fit(X_train, y_train)
LinearDiscriminant_GridSearchCV.best_params_
print("LinearDiscriminant best param = " + str(LinearDiscriminant_GridSearchCV.best_params_))

# Modèle RidgeClassifier
RidgeClassifier_parameters = {'alpha' : list(range(1,20)),
                              'fit_intercept' : [True, False],
                              'copy_X' : [True, False],
                              'tol' : [0.0001,0.0002,0.0003],
                              'solver' : ['auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga']}

RidgeClassifier_GridSearchCV = GridSearchCV(estimator = RidgeClassifier(), param_grid = RidgeClassifier_parameters, cv = 5, n_jobs=-1)
RidgeClassifier_GridSearchCV.fit(X_train, y_train)
RidgeClassifier_GridSearchCV.best_params_
print("RidgeClassifier best param = " + str(RidgeClassifier_GridSearchCV.best_params_))

# Modèle KNeighbors
KNeighborsClassifier_parameters = {#'n_neighbors' : list(range(1,50)),
                                   #'leaf_size' : list(range(1,30)), 
                                   'p':[1,2], 
                                   'algorithm' : ['auto', 'ball_tree', 'kd_tree', 'brute'], 
                                   'metric' : ['minkowski','euclidean','manhattan']}

KNeighborsClassifier_GridSearchCV = GridSearchCV(estimator = KNeighborsClassifier(), param_grid = KNeighborsClassifier_parameters, cv = 5, n_jobs=-1)
KNeighborsClassifier_GridSearchCV.fit(X_train, y_train)
KNeighborsClassifier_GridSearchCV.best_params_
print("KNeighborsClassifier best param = " + str(KNeighborsClassifier_GridSearchCV.best_params_))

# Modèle GaussianNB
GaussianNB_parameters = {'var_smoothing': np.logspace(0,-9, num=100)}
GaussianNB_GridSearchCV = GridSearchCV(estimator = GaussianNB(), param_grid = GaussianNB_parameters, cv = 5, n_jobs=-1)
GaussianNB_GridSearchCV.fit(X_train, y_train)
GaussianNB_GridSearchCV.best_params_

print("GaussianNB best param = " +str(GaussianNB_GridSearchCV.best_params_))

# Modèle DecisionTree
DecisionTreeClassifier_parameters = {#'max_features' : ['auto', 'sqrt', '“log2'],
                                     'max_depth': [2, 10, 15,18,20],
                                     'min_samples_leaf': [1, 2, 5, 10, 20, 50, 100],
                                     "min_samples_split": [2, 6, 20],
                                     'criterion': ["gini", "entropy"],
                                     'splitter' : ['best', 'random'],
                                     'min_samples_split' : np.linspace(0.1, 1.0, 5, endpoint=True).tolist()}

DecisionTreeClassifier_GridSearchCV = GridSearchCV(estimator = DecisionTreeClassifier(), param_grid = DecisionTreeClassifier_parameters, cv = 5, n_jobs=-1, verbose = 2)
DecisionTreeClassifier_GridSearchCV.fit(X_train, y_train)
DecisionTreeClassifier_GridSearchCV.best_params_
print("DecisionTreeClassifier best param = " +str(DecisionTreeClassifier_GridSearchCV.best_params_))

# Modèle SVC
SVC_parameters = {#'C': [1, 10, 50, 100, 200, 300],
                  'gamma': [0.0001, 0.001, 0.01, 0.1, 1],
                  'kernel': ['rbf', 'linear', 'poly', 'sigmoid'],
                  'probability': [True, False]}
                  
SVC_GridSearchCV = GridSearchCV(estimator = SVC(), param_grid = SVC_parameters, cv=5, n_jobs=-1, verbose=2)
SVC_GridSearchCV.fit(X_train, y_train)
SVC_GridSearchCV.best_params_
print("SVC best param = " +str(SVC_GridSearchCV.best_params_))

#Modèle SGD
SGD_parameters = {'loss' : ['hinge', 'log', 'modified_huber', 'squared_hinge', 'perceptron', 'squared_error', 'huber', 'epsilon_insensitive', 'squared_epsilon_insensitive'],
                  'penalty' : ['l1', 'l2', 'elasticnet'], 
                  'alpha' : [1e-5, 1e-4, 1e-3, 1e-2, 1e-1], 
                  'l1_ratio' : [0, 0.10, 0.15, 0.20, 0.30], 
                  'fit_intercept' : [True, False],
                  'tol' : [0.0001,0.0002,0.0003]}


SGD_GridSearchCV = GridSearchCV(estimator = SGDClassifier(), param_grid = SGD_parameters, cv=5, n_jobs=-1, verbose=2)
SGD_GridSearchCV.fit(X_train, y_train)
SGD_GridSearchCV.best_params_
print("SGD best param = " +str(SGD_GridSearchCV.best_params_))

# Modèle RandomForestClassifier
RandomForestClassifier_parameters = {'criterion' : ['gini', 'entropy'],
                                     'n_estimators' : [1,10,20,30, 100, 200, 400], 
                                     'min_samples_split' : [2,5,7,9,10], 
                                     'min_samples_leaf' : [1,2,5,7,9,10], 
                                     'max_features' : ['auto', 'sqrt', 'log2']}


RandomForestClassifier_GridSearchCV = GridSearchCV(estimator = RandomForestClassifier(), param_grid = RandomForestClassifier_parameters, cv=5, n_jobs=-1, verbose=2)
RandomForestClassifier_GridSearchCV.fit(X_train, y_train)
RandomForestClassifier_GridSearchCV.best_params_
coefficients = RandomForestClassifier_GridSearchCV.best_estimator_.feature_importances_
print("RandomForestClassifier best param = " +str(RandomForestClassifier_GridSearchCV.best_params_))

### ✅ CORRECTION

Dans le code original :  
❌ Pas de **pipeline** : la standardisation peut induire une fuite de données.  
❌ La recherche d’hyperparamètres et la comparaison des modèles ne sont pas organisées dans un schéma d’évaluation robuste.  
❌ Les hyperparamètres explorés conditionnent fortement le sur-/sous-apprentissage : prendre notamment garde à l'espace de recherche des paramètres qui contrôlent la régularisation, le nombre d'arbes pour RF, leur profondeur, le nombre de plus proches voisins pour kNN, etc etc.   


✅ Il vaut mieux intégrer la recherche des hyper-paramètres à l'évaluation des modèles (via une nested CV) sinon risque de fuite de données : on évalue les performances du modèle sur les données qui ont servi à définir les meilleurs hyper-paramètres pour ce modèle. 

<font size = 6> ✅ </font> 

In [ ]:
## CORRECTION
# ============================================================================
# CONFIGURATION DES PIPELINES ET GRID SEARCH
# ============================================================================
# NOTE IMPORTANTE SUR LE PIPELINE:
# Le Pipeline garantit que:
# 1. StandardScaler est fit UNIQUEMENT sur le train de chaque fold
# 2. Les données de validation de chaque fold sont transformées avec les 
#    paramètres appris sur le train (pas de data leakage)
# 3. Évite la contamination entre train et validation

# Définition des grilles de paramètres pour chaque modèle
# IMPORTANT: Les noms des paramètres doivent être préfixés par "classifier__"
# car ils sont dans un Pipeline

# Définition des grilles de paramètres pour chaque modèle
param_grids = {
    'LinearDiscriminant': {
        'classifier__solver': ['svd', 'lsqr', 'eigen'],
        'classifier__tol': [0.0001, 0.001, 0.01],
        'classifier__store_covariance': [True]
    },
    'Ridge': {        
        'classifier__alpha': [0.1, 1, 5, 10, 13, 20, 50],
        'classifier__fit_intercept': [True, False],
        'classifier__solver': ['auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag'],
        'classifier__tol': [0.0001, 0.001, 0.01]
    },
    'KNeighbors': {
        'classifier__n_neighbors': [3, 5, 7],
        'classifier__weights': ['uniform', 'distance'],
        'classifier__algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
        'classifier__p': [1, 2],  # 1=Manhattan, 2=Euclidean
        'classifier__metric': ['minkowski', 'euclidean', 'manhattan']
    },
    'GaussianNB': {
        'classifier__var_smoothing': np.logspace(-10, -5, 10)
    },
    'DecisionTree': {
        'classifier__criterion': ['gini', 'entropy'],
        'classifier__splitter': ['best', 'random'],
        'classifier__max_depth': [2, 10, 15],
        'classifier__min_samples_split': np.linspace(0.1, 1.0, 5, endpoint=True).tolist(),
        'classifier__min_samples_leaf': [1, 2, 5, 10, 20, 50, 100]
    },
    'SVC': { 
        'classifier__C': [0.01, 0.1, 1],
        'classifier__kernel': ['linear', 'rbf'],
        'classifier__gamma': ['scale', 0.001, 0.01, 0.1],
        'classifier__probability': [True]  # Obligatoire pour predict_proba
    },
    'SGD': {
        'classifier__loss': ['hinge', 'log', 'modified_huber', 'squared_hinge', 'perceptron', 'squared_error', 'huber', 'epsilon_insensitive', 'squared_epsilon_insensitive'],
        'classifier__penalty': ['l2', 'l1', 'elasticnet'],
        'classifier__alpha': [0.0001, 0.001, 0.01, 0.1],
        'classifier__l1_ratio': [0, 0.15, 0.2, 0.5, 0.8, 1],
        'classifier__fit_intercept': [True, False],
        'classifier__tol': [0.0001, 0.001, 0.01]
    },
    'RandomForest': {
        'classifier__n_estimators': [10,20, 30, 50],
        'classifier__criterion': ['gini', 'entropy'],
        'classifier__max_depth': [2, 5, 10, 15],
        'classifier__min_samples_split': [2, 5, 7, 10],
        'classifier__min_samples_leaf': [1,2,5,7,10],
        'classifier__max_features': ['sqrt']
    }
}


# Classes des modèles
models_classes = {
    'LinearDiscriminant': LinearDiscriminantAnalysis,
    'Ridge': RidgeClassifier,
    'KNeighbors': KNeighborsClassifier,
    'GaussianNB': GaussianNB,
    'DecisionTree': DecisionTreeClassifier,
    'SVC': SVC,
    'SGD': SGDClassifier,
    'RandomForest': RandomForestClassifier
}

## Evaluation des modèles

### 🚫 CODE ORIGINAL

<font size = 6> 🚫 </font>  

In [ ]:
## PUBLI ORIGINALE
# ============================================================================
# ÉVALUATION DES MODELES (ENTRAINEMENT vs TEST)
# ============================================================================
List_of_models = [LinearDiscriminantAnalysis(solver = 'svd', store_covariance = True, tol = 0.0001),
                  RidgeClassifier(alpha = 13, copy_X = True, fit_intercept = False, solver = 'auto', tol = 0.0001),
                  KNeighborsClassifier(algorithm = 'auto', metric = 'minkowski', p = 1),
                  GaussianNB(var_smoothing = 0.2848035868435802),
                  DecisionTreeClassifier(criterion = 'gini', max_depth = 2, min_samples_leaf = 100, min_samples_split = 0.1, splitter = 'best'),
                  SVC(gamma = 0.1, kernel = 'sigmoid', probability = True),
                  SGDClassifier(alpha = 0.001, fit_intercept = False, l1_ratio = 0.2, loss = 'huber', penalty = 'l2', tol = 0.0002),
                  RandomForestClassifier(criterion = 'entropy', max_features = 'sqrt', min_samples_leaf = 2, min_samples_split = 5, n_estimators = 30)]

List_of_models_for_graph = ["LinearDiscriminant", "Ridge", "KNeighbors", "GaussianNB", "DecisionTree", "SVC", "SGD", "RandomForestClassifier"]

df_results_publi = pd.DataFrame(index = ["Score entrainement", "Score de prédiction", "MAE", "RMSE", "median absolute error"], columns = ["LinearDiscriminant"])

custom_palette = [sns.xkcd_rgb["windows blue"], sns.xkcd_rgb["pale red"], sns.xkcd_rgb["green blue"], "orange", sns.xkcd_rgb["blue"],sns.xkcd_rgb["sunny yellow"], sns.xkcd_rgb["warm purple"], sns.xkcd_rgb["medium green"], "brown", "teal", "black"] 
sns.set_palette(custom_palette)

#custom_palette = sns.color_palette("tab10")

for i in range(len(List_of_models)):
  model_class = List_of_models[i] 
  model_class.fit(X_train, y_train)
  y_pred = model_class.predict(X_test)
  results_classification = np.array([model_class.score(X_train,y_train), model_class.score(X_test,y_test), mean_absolute_error(y_test,y_pred), np.sqrt(mean_squared_error(y_test,y_pred)), median_absolute_error(y_test,y_pred)])
  df_results_publi[List_of_models_for_graph[i]] = results_classification

  df_graph = df_results_publi.transpose()
fig, axs = plt.subplots(1, 2, figsize=(12, 6))
ax1 = sns.barplot(x=df_graph.index, y=df_graph["Score entrainement"].values, data=df_graph, ax=axs[0], palette = custom_palette)
ax2 = sns.barplot(x=df_graph.index, y=df_graph["Score de prédiction"].values, data=df_graph, ax=axs[1], palette = custom_palette)
ax1.set_xticklabels(ax1.get_xticklabels(),rotation=45)
#ax1.set_title('Efficacité des modèles pour la prédiction du gamma moyen \n pour toutes les localisations')
ax1.set_ylabel('Training score')
ax1.set(ylim=(0, 1))
ax2.set_xticklabels(ax2.get_xticklabels(),rotation=45)
#ax2.set_title('Efficacité des modèles pour la prédiction du gamma moyen \n pour toutes les localisations')
ax2.set_ylabel('Validation score')
ax2.set(ylim=(0, 1))

plt.tight_layout()
fig.savefig("Performance modèles ML trois metriques", dpi=400)

In [ ]:
df_results_publi

### ✅ CORRECTION

<font size = 6> ⚠️ </font> 

$$
\mathrm{MAE} = \mathrm{mean}\left(\left|y - y_{\mathrm{pred}}\right|\right)
$$

$$
\mathrm{RMSE} = \sqrt{\mathrm{mean}\left((y - y_{\mathrm{pred}})^2\right)}
$$

Sur une **classification binaire**, calculer MAE/RMSE sur des **classes prédites (0/1)** est possible mathématiquement, mais ce sont des métriques **peu informatives** pour comparer des classifieurs.

- Avec des labels 0/1, le **MAE** revient essentiellement à une mesure liée au **taux d’erreur** (1-accuracy).
- Ces métriques ne rendent pas bien compte du compromis **sensibilité / spécificité**, ni du comportement selon le **seuil de décision**.

✅ En classification binaire, on préfère généralement :  
- AUC-ROC / AUC-PR,  
- sensibilité / spécificité,  
- balanced accuracy,  
- F1-score (selon le contexte),  
- et éventuellement le **Brier score** si l’on évalue des **probabilités**.

Dans le code original :  

❌ Les **scores par défaut** de scikit-learn sont utilisés sans préciser ce qu’ils représentent : pour la plupart des classifieurs, il s’agit de l’**accuracy**.  
➡️ L’accuracy n’est pas forcément “fausse”, mais **elle est insuffisante seule** pour juger un classifieur, surtout si les coûts des faux positifs / faux négatifs sont asymétriques.

❌ Le tableau ne précise pas explicitement à quoi correspondent les scores affichés (train ? test ? métrique exacte ?), ce qui nuit à l’interprétation.

❌ Le **score d’entraînement** (calculé sur l’ensemble des données d’entraînement) est présenté comme un indicateur de comparaison :  
➡️ ce score est souvent **optimiste** et n’est pas comparable à une performance obtenue en validation croisée.  
✅ Pour comparer des modèles, il vaut mieux utiliser les scores obtenus sur les **folds de validation de CV** (moyenne ± écart-type).

❌ Le **score sur les données test** est utilisé trop tôt dans le processus de sélection du modèle.  
➡️ Le test set doit rester un **jeu de données final**, utilisé une seule fois (ou très peu) pour l’évaluation finale.

❌ `y_pred` est généré avec `predict()` puis réutilisé pour des métriques qui nécessitent des **scores continus / probabilités** (ex. ROC-AUC, PR-AUC). Le code ne renvoie pas d'erreur mais le calcul est faux.   
✅ Pour ces métriques, il faut utiliser `predict_proba()` (ou `decision_function()` selon le modèle).

❌ Sans validation croisée, on ne mesure pas la **variabilité** des performances liée au split (instabilité du modèle / de l’échantillonnage).

✅ Il faut séparer clairement  
1) **sélection / tuning sur entraînement (CV)**  
2) **évaluation finale sur test indépendant**

<font size = 6> ✅ </font> 

Pour comparer des modèles de façon plus rigoureuse :  
✅ Utiliser des **Pipelines** (prétraitement + modèle) pour éviter les fuites de données.  
✅ Comparer les modèles via une **validation croisée** sur les données d’entraînement.  
✅ Définir **explicitement** la/les métrique(s) de comparaison (AUC, balanced accuracy, sensibilité, etc.).  
✅ Conserver le **jeu de test** pour l’évaluation finale uniquement.

In [ ]:
print("\n" + "=" * 80)
print("GRID SEARCH AVEC PIPELINE (StandardScaler + Classifier)")
print("Métrique d'optimisation: ROC-AUC")
print("=" * 80)

best_pipelines = {}
grid_search_results = {}

cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for model_name in param_grids.keys():
    print(f"\n{'='*60}")
    print(f"Modèle: {model_name}")
    print(f"{'='*60}")
    
    # Pipeline : scaling + classifieur
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', models_classes[model_name]())
    ])
    
    # GridSearchCV (optimisation sur ROC-AUC)
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[model_name],
        scoring='roc_auc',
        cv=cv_splitter,
        n_jobs=-1,
        verbose=0,
        return_train_score=True,  # nécessaire pour récupérer mean/std train
        refit=True
    )
    
    print("Recherche en cours...", end=" ")
    try:
        grid_search.fit(X_train_corr, y_train_corr)
        print("OK")
    except Exception as e:
        print(f"Erreur: {e}")
        continue
    
    # Pipeline final refitté avec les meilleurs hyperparamètres
    best_pipelines[model_name] = grid_search.best_estimator_
    
    # Résultats CV
    cv_results = grid_search.cv_results_
    best_index = grid_search.best_index_
    
    # Moyennes et écarts-types des folds pour la meilleure combinaison d'hyperparamètres
    mean_train_auc = cv_results['mean_train_score'][best_index]
    std_train_auc  = cv_results['std_train_score'][best_index]
    mean_val_auc   = cv_results['mean_test_score'][best_index]   # folds validation CV
    std_val_auc    = cv_results['std_test_score'][best_index]
    
    # Stockage MINIMAL (sans scores par fold, sans objet grid_search complet)
    grid_search_results[model_name] = {
        'best_params': grid_search.best_params_,   # optionnel (peut être retiré)
        'mean_train_auc': mean_train_auc,
        'std_train_auc': std_train_auc,
        'mean_val_auc': mean_val_auc,
        'std_val_auc': std_val_auc
    }
    
    # Affichage
    print(f"Meilleurs paramètres: {grid_search.best_params_}")
    print(f"AUC CV (train): {mean_train_auc:.4f} ± {std_train_auc:.4f}")
    print(f"AUC CV (val):   {mean_val_auc:.4f} ± {std_val_auc:.4f}")

In [ ]:
# ============================================================================
# SYNTHÈSE + BARPLOTS (à partir des moyennes/std des meilleurs hyperparamètres)
# ============================================================================

if len(grid_search_results) == 0:
    raise ValueError("Aucun modèle n'a pu être entraîné.")

# Construction du DataFrame récapitulatif à partir du dictionnaire grid_search_results
df_graph = pd.DataFrame({
    "Modèle": list(grid_search_results.keys()),
    "AUC CV train": [grid_search_results[m]["mean_train_auc"] for m in grid_search_results.keys()],
    "STD train":    [grid_search_results[m]["std_train_auc"]  for m in grid_search_results.keys()],
    "AUC CV val":   [grid_search_results[m]["mean_val_auc"]   for m in grid_search_results.keys()],
    "STD val":      [grid_search_results[m]["std_val_auc"]    for m in grid_search_results.keys()],
})

# (Optionnel) Tri par performance de validation décroissante
df_graph = df_graph.sort_values("AUC CV val", ascending=False).reset_index(drop=True)

# Palette de couleurs
custom_palette = sns.color_palette("tab10", n_colors=len(df_graph))
sns.set_palette(custom_palette)

# Création de la figure avec 2 sous-graphes (train / validation)
fig, axs = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# ----------------------------------------------------------------------------
# Barplot AUC CV train
# ----------------------------------------------------------------------------
sns.barplot(
    x="Modèle",
    y="AUC CV train",
    data=df_graph,
    ax=axs[0],
    palette=custom_palette,
    errorbar=None  # on ajoute les barres d'erreur manuellement
)

# Barres d'erreur = écart-type CV sur les folds train
axs[0].errorbar(
    x=np.arange(len(df_graph)),
    y=df_graph["AUC CV train"].values,
    yerr=df_graph["STD train"].values,
    fmt='none',
    ecolor='black',
    elinewidth=1,
    capsize=4
)

axs[0].set_ylabel("AUC CV train")
axs[0].set_xlabel("")
axs[0].set_ylim(0, 1)
plt.setp(axs[0].get_xticklabels(), rotation=45, ha='right')

# ----------------------------------------------------------------------------
# Barplot AUC CV validation
# ----------------------------------------------------------------------------
sns.barplot(
    x="Modèle",
    y="AUC CV val",
    data=df_graph,
    ax=axs[1],
    palette=custom_palette,
    errorbar=None
)

# Barres d'erreur = écart-type CV sur les folds validation
axs[1].errorbar(
    x=np.arange(len(df_graph)),
    y=df_graph["AUC CV val"].values,
    yerr=df_graph["STD val"].values,
    fmt='none',
    ecolor='black',
    elinewidth=1,
    capsize=4
)

axs[1].set_ylabel("AUC CV validation")
axs[1].set_xlabel("")
axs[1].set_ylim(0, 1)
plt.setp(axs[1].get_xticklabels(), rotation=45, ha='right')

# Titre global
plt.suptitle("Comparaison des modèles (GridSearchCV + Pipeline + ROC-AUC)", y=1.02)

# Ajustement des marges
plt.tight_layout()

# Affichage
plt.show()

# Tableau récapitulatif
display(df_graph)

In [ ]:
# ============================================================================
# SÉLECTION DU MEILLEUR MODÈLE (sur la moyenne AUC CV de validation)
# ============================================================================

# Tableau récapitulatif (pratique pour trier/afficher)
df_selection = (
    pd.DataFrame(grid_search_results)
    .T
    .reset_index()
    .rename(columns={"index": "Modèle"})
)


# Tri :
# 1) meilleure AUC validation moyenne (descendant)
# 2) plus faible std validation (ascendant) = modèle plus stable
df_selection = df_selection.sort_values(
    by=["mean_val_auc", "std_val_auc"],
    ascending=[False, True]
).reset_index(drop=True)

# Nom du meilleur modèle
best_model_name = df_selection.loc[0, "Modèle"]

# Pipeline déjà refitté sur tout X_train_corr / y_train_corr avec les meilleurs hyperparams
best_pipeline = best_pipelines[best_model_name]

# Infos utiles
best_model_info = grid_search_results[best_model_name]

print("\n" + "=" * 80)
print("MEILLEUR MODÈLE SÉLECTIONNÉ")
print("=" * 80)
print(f"Modèle retenu: {best_model_name}")
print(f"Meilleurs hyperparamètres: {best_model_info['best_params']}")
print(f"AUC CV (train): {best_model_info['mean_train_auc']:.4f} ± {best_model_info['std_train_auc']:.4f}")
print(f"AUC CV (val):   {best_model_info['mean_val_auc']:.4f} ± {best_model_info['std_val_auc']:.4f}")



## EVALUATION DES PERFORMANCES DU MODELE SELECTIONNE

### Aire sous la courbe ROC

#### 🚫 CODE ORIGINAL

<font size = 6> 🚫 </font>  

In [ ]:
## PUBLI ORIGINALE
# ============================================================================
# MEILLEUR MODELE = RFC
# ============================================================================
RFC = RandomForestClassifier(criterion = 'entropy', max_features = 'sqrt', min_samples_leaf = 2, min_samples_split = 5, n_estimators = 30)
RFC.fit(X_train, y_train)

pred = RFC.predict(X)
auc_publi = roc_auc_score(Y, pred)
print("AUC = " + str(round(auc_publi,4)))

#### ✅ CORRECTION

❌ `pred` est calculé sur **toute la base X** (train + test).  
➡️ Cela biaise fortement l’évaluation : le modèle a déjà vu une grande partie de ces données pendant l’entraînement, donc la performance est **artificiellement surestimée**.

❌ L’**AUC-ROC** est calculée à partir de `predict()` (classes 0/1) au lieu de scores continus (`predict_proba()` ou `decision_function()`).  
➡️ La courbe ROC décrit l’évolution du TPR/FPR selon **plusieurs seuils**.  
Avec `predict()`, on fixe implicitement un seuil (souvent 0.5) et on obtient essentiellement **un point de fonctionnement**, pas une vraie courbe ROC exploitable.

⚠️ C’est une erreur très piégeuse : le code s’exécute sans message d’erreur, mais l’interprétation est fausse.

✅ Pour une ROC/AUC :
- évaluer sur les **données test uniquement** (ou sur des prédictions hors-fold en CV),
- utiliser `predict_proba()[:, 1]` (ou `decision_function()` selon le modèle).

#####  <font size = 6> ⚠️ </font>   **Avec le même modèle que la publi**

In [ ]:
## CORRECTION EN UTILISANT LE MEME modèle QUE LA PUBLI
RFC = RandomForestClassifier(criterion = 'entropy', max_features = 'sqrt', min_samples_leaf = 2, min_samples_split = 5, n_estimators = 30, random_state=2026)
RFC.fit(X_train, y_train)

pred_proba = RFC.predict_proba(X)[:, 1]  # Probabilité de la classe 1
pred_proba_test = RFC.predict_proba(X_test)[:, 1]  # Probabilité de la classe 1
auc = roc_auc_score(Y, pred_proba)
auc_test = roc_auc_score(y_test, pred_proba_test)
print("AUC FAUX calculé à partir de predict, sur toute la base de données (à ne pas faire) comme dans la publi = " + str(round(auc_publi,4)))
print("AUC calculé correctement, mais sur toute la base de données comme dans la publi (à ne jamais faire) = " + str(round(auc,4)))
print("AUC calculé correctement et seulement sur des données inconnues (test) = " + str(round(auc_test,4)))
print("En calculant l'AUC sur toute la base : surestimation des performances de " + str(round(100*(auc-auc_test)/auc_test,1)) + " % ! ")
print("Pas étonnant, puisque le modèle a déjà vu 80% de la base !")

##### <font size = 6> ✅ </font>   **Avec le modèle corrigé**

In [ ]:
## CORRECTION EN UTILISANT LE MODELE CORRIGE

best_pipeline.fit(X_train_corr, y_train_corr)

pred_proba_test = best_pipeline.predict_proba(X_test_corr)[:, 1]  # Probabilité de la classe 1
auc_test = roc_auc_score(y_test_corr, pred_proba_test)
print("AUC calculé correctement et seulement sur des données inconnues (test) = " + str(round(auc_test,4)))

### Courbe ROC

#### 🚫 CODE ORIGINAL 

<font size = 6> 🚫 </font>  

In [ ]:
## PUBLI ORIGINALE
y = Y
#y = y[:,1]
yhat = RFC.predict(X)

#yhat = yhat[:,1]

from numpy import argmax
from numpy import sqrt

# calculate roc curves
fpr, tpr, thresholds = roc_curve(y, yhat)
# calculate the g-mean for each threshold
gmeans = sqrt(tpr * (1-fpr))
# locate the index of the largest g-mean
ix = argmax(gmeans)
print('Best Threshold=%f, G-Mean=%.3f' % (thresholds[ix], gmeans[ix]))
auc = roc_auc_score(y, yhat)
print('AUC=%f' %auc)
# plot the roc curve for the model
plt.plot([0,1], [0,1], linestyle='--', label='No Skill')
plt.plot(fpr, tpr, marker='.', label='RandomForestClassifier')
plt.scatter(fpr[ix], tpr[ix], marker='o', color='black', label='Best')
plt.text(0.5,0.8,'AUC=%f' %round(auc,4) ,horizontalalignment='center',
     verticalalignment='center', fontsize=10, color='black')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
# show the plot
plt.tight_layout()

plt.savefig("RFC one class", dpi=400)
plt.show()

#### ✅ CORRECTION

❌ Ici encore, l’**AUC-ROC** est calculée avec `predict()` au lieu de `predict_proba()` / `decision_function()`.  
✅ Pour une ROC correcte, il faut des **scores continus**, pas des classes binaires.

❌ L’allure de la courbe (seulement 3 points) devrait alerter : une ROC calculée correctement est généralement plus “continue” (selon la taille de l’échantillon et la granularité des scores).

##### <font size = 6> ⚠️ </font>  **Sur le même principe et même modèle que la publi originale mais sans les erreurs de code et donc la vraie allure de la courbe :**  
❌ Attention le résultat est tout de même faux car évalué sur toute la base

In [ ]:
## CORRECTION AVEC AFFICHAGE CORRECT DE LA COURBE ROC (MAIS RESULTAT TROMPEUR CAR EVALUE SUR BASE ENTIERE = TRAIN + TEST)
y = Y
yhat = RFC.predict_proba(X)[:, 1] 

from numpy import argmax
from numpy import sqrt

# calculate roc curves
fpr, tpr, thresholds = roc_curve(y, yhat)
# calculate the g-mean for each threshold
gmeans = sqrt(tpr * (1-fpr))
# locate the index of the largest g-mean
ix = argmax(gmeans)
print('Best Threshold=%f, G-Mean=%.3f' % (thresholds[ix], gmeans[ix]))
auc = roc_auc_score(y, yhat)
print('AUC=%f' %auc)
# plot the roc curve for the model
plt.plot([0,1], [0,1], linestyle='--', label='No Skill')
plt.plot(fpr, tpr, marker='.', label='RandomForestClassifier')
plt.scatter(fpr[ix], tpr[ix], marker='o', color='black', label='Best')
plt.text(0.5,0.8,'AUC=%f' %round(auc,4) ,horizontalalignment='center',
     verticalalignment='center', fontsize=10, color='black')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
# show the plot
plt.tight_layout()

plt.savefig("RFC one class", dpi=400)
plt.show()

**Une courbe ROC presque parfaite : le modèle semble excellent.**  

⚠️ Ici, cette performance est trompeuse car l’évaluation inclut des données déjà vues pendant l’entraînement (train + test mélangés dans l’évaluation).

✅ Principe fondamental : les performances finales doivent être estimées sur des **données réellement nouvelles** pour le modèle (jamais vues pendant l’entraînement, le tuning, la sélection de variables, ni le choix du seuil).

##### <font size = 6> ✅ </font>  **Sur les données test avec le même modèle que la publi originale**

In [ ]:
## CORRECTION AVEC AFFICHAGE CORRECT DE LA COURBE ROC ET DONNEES TEST INCONNUES
y = y_test
#y = y[:,1]
yhat = RFC.predict_proba(X_test)[:, 1] 
#yhat = yhat[:,1]

from numpy import argmax
from numpy import sqrt

# calculate roc curves
fpr, tpr, thresholds = roc_curve(y, yhat)
# calculate the g-mean for each threshold
gmeans = sqrt(tpr * (1-fpr))
# locate the index of the largest g-mean
ix = argmax(gmeans)
print('Best Threshold=%f, G-Mean=%.3f' % (thresholds[ix], gmeans[ix]))
auc = roc_auc_score(y, yhat)
print('AUC=%f' %auc)
# plot the roc curve for the model
plt.plot([0,1], [0,1], linestyle='--', label='No Skill')
plt.plot(fpr, tpr, marker='.', label='RandomForestClassifier')
plt.scatter(fpr[ix], tpr[ix], marker='o', color='black', label='Best')
plt.text(0.5,0.8,'AUC=%f' %round(auc,4) ,horizontalalignment='center',
     verticalalignment='center', fontsize=10, color='black')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
# show the plot
plt.tight_layout()

plt.savefig("RFC one class", dpi=400)
plt.show()

 <font size = 6> ⚠️</font> **La courbe ROC est déjà beaucoup plus réaliste lorsqu’elle est calculée sur les données test uniquement.**

⚠️ Mais l’estimation reste encore potentiellement **optimiste** car :
- le prétraitement (ex. standardisation) a été ajusté avant le split ;
- le modèle final a été choisi en regardant ses performances sur le test set ;
- des décisions méthodologiques ont été prises après inspection répétée des résultats test.

✅ Le test set doit rester un **jeu d’évaluation final**.  
La sélection du modèle et des hyperparamètres doit se faire sur les données d’entraînement (via CV).

##### <font size = 6> ✅ </font>  **Sur les données test avec le modèle corrigé**

In [ ]:
## CORRECTION AVECLE NOUVEAU MODELE
y = y_test_corr
#y = y[:,1]
yhat = best_pipeline.predict_proba(X_test_corr)[:, 1] 
#yhat = yhat[:,1]

from numpy import argmax
from numpy import sqrt

# calculate roc curves
fpr, tpr, thresholds = roc_curve(y, yhat)
# calculate the g-mean for each threshold
gmeans = sqrt(tpr * (1-fpr))
# locate the index of the largest g-mean
ix_corr = argmax(gmeans)
print('Best Threshold=%f, G-Mean=%.3f' % (thresholds[ix_corr], gmeans[ix_corr]))
auc = roc_auc_score(y, yhat)
print('AUC=%f' %auc)
# plot the roc curve for the model
plt.plot([0,1], [0,1], linestyle='--', label='No Skill')
plt.plot(fpr, tpr, marker='.', label='Best pipeline')
plt.scatter(fpr[ix], tpr[ix], marker='o', color='black', label='Best (G-Mean)')
plt.text(0.5,0.8,'AUC=%f' %round(auc,4) ,horizontalalignment='center',
     verticalalignment='center', fontsize=10, color='black')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
# show the plot
plt.tight_layout()

plt.savefig("RFC one class", dpi=400)
plt.show()

⚠️ **Si le modèle avait été construit correctement, il aurait été plus performant que le modèle proposé dans l'article dont la construction et l'évaluation sont biaisées par de la fuite de donnée.** 

⚠️ Le modèle “corrigé” peut paraître moins spectaculaire sur certaines évaluations, mais il fournit une estimation **beaucoup plus crédible** des performances réelles.

✅ Un pipeline construit sans fuite de données et évalué correctement est préférable à un modèle affichant de très bonnes performances mais obtenues avec un protocole biaisé.

### Matrice de confusion

#### 🚫 CODE ORIGINAL 

<font size = 6> 🚫 </font>  

In [ ]:
## PUBLI ORIGINALE (affiché avec TensorFlow mais le résultat est le même) : EVALUATION SUR TRAIN + TEST
pred = RFC.predict(X)
conf_matrix_RFC = confusion_matrix(y_true=Y, y_pred=pred)

# Extraction des valeurs
tn, fp, fn, tp = conf_matrix_RFC.ravel()

# Calcul des métriques
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
accuracy = (tp + tn) / (tp + tn + fp + fn)

print(f"\nTP = {tp}, TN = {tn}, FP = {fp}, FN = {fn}")
print(f"Sensitivity = {sensitivity:.4f}")
print(f"Specificity = {specificity:.4f}")
print(f"Accuracy = {accuracy:.4f}")
# ============================================================================
# VISUALISATION DE LA MATRICE DE CONFUSION
# ============================================================================
fig, ax = plt.subplots(figsize=(8, 6))

ax_rfc = sns.heatmap(
    conf_matrix_RFC, 
    annot=True, 
    fmt='d',
    cmap='Blues',
    cbar=True,
    square=True,
    xticklabels=['Non-Conforme (0)', 'Conforme (1)'],
    yticklabels=['Non-Conforme (0)', 'Conforme (1)'],
    ax=ax
)

ax_rfc.set_title(
    f'RandomForestClassifier, AUC = {round(auc, 4)}',
    fontsize=14,
    fontweight='bold',
    pad=15
)
ax_rfc.set_ylabel('Réel', fontsize=12, fontweight='bold')
ax_rfc.set_xlabel('Prédiction', fontsize=12, fontweight='bold')

# Annoter avec les pourcentages
total = conf_matrix_RFC.sum()
for i in range(2):
    for j in range(2):
        percentage = conf_matrix_RFC[i, j] / total * 100
        ax.text(
            j + 0.5, i + 0.7, 
            f'({percentage:.1f}%)',
            ha="center", va="center", 
            color="red", fontsize=10
        )

plt.tight_layout()

plt.show()

#### ✅ CORRECTION 

❌ L’évaluation est encore réalisée sur **toute la base de données** et non sur les seules données test.  
➡️ La matrice de confusion est donc **faussement optimiste**.

❌ Un contrôle simple aurait dû alerter : la somme des effectifs de la matrice correspond à la **totalité de la base**, alors qu’elle devrait correspondre uniquement au **nombre d’exemples du test set**.

✅ Toujours vérifier :
- sur quel jeu la matrice est calculée (train / validation / test),
- et que la somme des cases correspond bien au nombre d’exemples attendu.

##### <font size = 6> ⚠️ </font>  **Avec le même modèle que la publi mais sur les données test seulement :**

In [ ]:
## CORRECTION : EVALUATION SUR TEST SEULEMENT
pred_test = RFC.predict(X_test)
conf_matrix_RFC = confusion_matrix(y_true=y_test, y_pred=pred_test)

# Extraction des valeurs
tn, fp, fn, tp = conf_matrix_RFC.ravel()

# Calcul des métriques
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
accuracy = (tp + tn) / (tp + tn + fp + fn)

print(f"\nTP = {tp}, TN = {tn}, FP = {fp}, FN = {fn}")
print(f"Sensitivity = {sensitivity:.4f}")
print(f"Specificity = {specificity:.4f}")
print(f"Accuracy = {accuracy:.4f}")

# ============================================================================
# VISUALISATION DE LA MATRICE DE CONFUSION
# ============================================================================
fig, ax = plt.subplots(figsize=(8, 6))

ax_rfc = sns.heatmap(
    conf_matrix_RFC, 
    annot=True, 
    fmt='d',
    cmap='Blues',
    cbar=True,
    square=True,
    xticklabels=['Non-Conforme (0)', 'Conforme (1)'],
    yticklabels=['Non-Conforme (0)', 'Conforme (1)'],
    ax=ax
)

ax_rfc.set_title(
    f'RandomForestClassifier, AUC = {round(auc, 4)}',
    fontsize=14,
    fontweight='bold',
    pad=15
)
ax_rfc.set_ylabel('Réel', fontsize=12, fontweight='bold')
ax_rfc.set_xlabel('Prédiction', fontsize=12, fontweight='bold')

# Annoter avec les pourcentages
total = conf_matrix_RFC.sum()
for i in range(2):
    for j in range(2):
        percentage = conf_matrix_RFC[i, j] / total * 100
        ax.text(
            j + 0.5, i + 0.7, 
            f'({percentage:.1f}%)',
            ha="center", va="center", 
            color="red", fontsize=10
        )

plt.tight_layout()
plt.savefig("Matrice_de_confusion_RFC.png", dpi=400, bbox_inches='tight')
print("\n✓ Figure sauvegardée : Matrice_de_confusion_RFC.png")
plt.show()

##### <font size = 6> ✅ </font>  **Avec le nouveau modèle**

In [ ]:
# ----------------------------
# Données
# ----------------------------
y = y_test_corr
yhat = best_pipeline.predict_proba(X_test_corr)[:, 1]

# ----------------------------
# ROC + seuil optimal (G-Mean)
# ----------------------------
fpr, tpr, thresholds = roc_curve(y, yhat)

gmeans = np.sqrt(tpr * (1 - fpr))
ix = np.argmax(gmeans)

best_thresh = thresholds[ix]
auc = roc_auc_score(y, yhat)

print(f"Best Threshold={best_thresh:.6f}, G-Mean={gmeans[ix]:.3f}")
print(f"AUC={auc:.6f}")

# ============================================================================
# MATRICE DE CONFUSION AVEC LE SEUIL OPTIMAL
# ============================================================================
# Binarisation avec le seuil optimal
pred_test_thresh = (yhat >= best_thresh).astype(int)

conf_matrix_bestpipeline = confusion_matrix(y_true=y, y_pred=pred_test_thresh)
tn, fp, fn, tp = conf_matrix_bestpipeline.ravel()

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0

print(f"\nTP = {tp}, TN = {tn}, FP = {fp}, FN = {fn}")
print(f"Sensitivity = {sensitivity:.4f}")
print(f"Specificity = {specificity:.4f}")
print(f"Accuracy = {accuracy:.4f}")

# ============================================================================
# VISUALISATION (heatmap) — correction des variables / titres / sauvegarde
# ============================================================================
fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(
    conf_matrix_bestpipeline,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=True,
    square=True,
    xticklabels=["Non-Conforme (0)", "Conforme (1)"],
    yticklabels=["Non-Conforme (0)", "Conforme (1)"],
    ax=ax
)

ax.set_title(
    f"Best pipeline (seuil G-Mean={best_thresh:.3f}) — AUC={auc:.4f}",
    fontsize=14,
    fontweight="bold",
    pad=15
)
ax.set_ylabel("Réel", fontsize=12, fontweight="bold")
ax.set_xlabel("Prédiction", fontsize=12, fontweight="bold")

# Pourcentages par case
total = conf_matrix_bestpipeline.sum()
for i in range(2):
    for j in range(2):
        percentage = (conf_matrix_bestpipeline[i, j] / total * 100) if total > 0 else 0.0
        ax.text(
            j + 0.5, i + 0.75,
            f"({percentage:.1f}%)",
            ha="center", va="center",
            color="red", fontsize=10
        )

plt.tight_layout()

plt.show()


<font size = 6> 💡 </font> Les résultats de la matrice de confusion dépendent du **seuil de décision** (0.5 par défaut avec `predict()` dans beaucoup de classifieurs probabilistes).

✅ Il est possible d’ajuster ce seuil (par exemple via l’**indice de Youden**) pour privilégier la sensibilité, la spécificité, ou un compromis adapté au contexte clinique.

⚠️ Ce seuil doit être choisi à partir des **données d’entraînement** (idéalement via validation croisée ou sur un jeu de validation dédié), **pas à partir du test set**.

<font size = 6> ⚠️ </font> L’exemple ci-dessous est illustré sur le **jeu de test**.

C’est utile pédagogiquement pour montrer l’effet du seuil, mais **ce n’est pas une bonne pratique** si l’objectif est d’estimer une performance finale non biaisée.

✅ En pratique, le seuil de décision doit être défini :
- sur les données d’entraînement (via CV),
- ou sur un jeu de validation séparé,
puis appliqué **tel quel** au jeu de test final.

In [ ]:
# ============================================================================
# MATRICE DE CONFUSION : RESULTATS EN FONCTION DU SEUIL DE DECISION 
# ============================================================================

y_pred_proba_test = best_pipeline.predict_proba(X_test_corr)[:, 1]

if y_pred_proba_test is not None:
    print("\n" + "=" * 80)
    print("GÉNÉRATION DE LA COURBE ROC")
    print("=" * 80)

    # ROC
    fpr, tpr, thresholds = roc_curve(y_test_corr, y_pred_proba_test)

    # Seuil optimal (Youden)
    youden = tpr - fpr
    optimal_idx = np.argmax(youden)
    optimal_threshold = thresholds[optimal_idx]
    optimal_fpr = fpr[optimal_idx]
    optimal_tpr = tpr[optimal_idx]

    # AUC recalculée (sécurisé)
    auc_test = roc_auc_score(y_test_corr, y_pred_proba_test)

    print(f"Seuil optimal (Youden's index) : {optimal_threshold:.4f}")
    print(f"  FPR au seuil optimal : {optimal_fpr:.4f}")
    print(f"  TPR au seuil optimal : {optimal_tpr:.4f}")
    print(f"  AUC : {auc_test:.4f}")

    # Plot ROC
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.plot(fpr, tpr, lw=3, label=f'ROC curve (AUC = {auc_test:.4f})')
    ax.plot([0, 1], [0, 1], lw=2, linestyle='--', label='Random classifier (AUC = 0.5)')
    ax.plot(
        optimal_fpr, optimal_tpr, 'o', markersize=12,
        label=f'Optimal threshold = {optimal_threshold:.3f}',
        markeredgecolor='black', markeredgewidth=1.5
    )

    ax.annotate(
        f'Threshold = {optimal_threshold:.3f}\nTPR = {optimal_tpr:.3f}\nFPR = {optimal_fpr:.3f}',
        xy=(optimal_fpr, optimal_tpr),
        xytext=(min(optimal_fpr + 0.15, 0.85), max(optimal_tpr - 0.15, 0.05)),
        fontsize=10,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.85),
        arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0.2', lw=2)
    )

    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=13, fontweight='bold')
    ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=13, fontweight='bold')
    ax.set_title(f'Courbe ROC - {best_model_name}\nTest Set', fontsize=14, fontweight='bold', pad=15)
    ax.legend(loc="lower right", fontsize=11, framealpha=0.9)
    ax.grid(alpha=0.3, linestyle='--')
    plt.tight_layout()
    plt.show()

    # ============================================================================
    # COMPARAISON SEUIL 0.5 vs SEUIL OPTIMAL (TP/TN/FP/FN CORRECTS)
    # ============================================================================
    print("\n" + "=" * 80)
    print("COMPARAISON SEUIL PAR DÉFAUT vs SEUIL OPTIMAL")
    print("=" * 80)

    # --- Prédictions binaires cohérentes avec les seuils ---
    y_pred_default = (y_pred_proba_test >= 0.5).astype(int)
    y_pred_optimal = (y_pred_proba_test >= optimal_threshold).astype(int)

    # --- Matrices de confusion recalculées ---
    conf_matrix_default = confusion_matrix(y_test_corr, y_pred_default)
    tn_def, fp_def, fn_def, tp_def = conf_matrix_default.ravel()

    conf_matrix_optimal = confusion_matrix(y_test_corr, y_pred_optimal)
    tn_opt, fp_opt, fn_opt, tp_opt = conf_matrix_optimal.ravel()

    # --- Métriques (fonction utilitaire) ---
    def _metrics_from_cm(tn, fp, fn, tp):
        total = tn + fp + fn + tp
        accuracy = (tp + tn) / total if total > 0 else 0.0
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        f1 = (2 * precision * sensitivity / (precision + sensitivity)) if (precision + sensitivity) > 0 else 0.0
        return accuracy, sensitivity, specificity, precision, f1

    accuracy_def, sensitivity_def, specificity_def, precision_def, f1_def = _metrics_from_cm(
        tn_def, fp_def, fn_def, tp_def
    )
    accuracy_opt, sensitivity_opt, specificity_opt, precision_opt, f1_opt = _metrics_from_cm(
        tn_opt, fp_opt, fn_opt, tp_opt
    )

    print(f"Seuil 0.5:   TP={tp_def}, TN={tn_def}, FP={fp_def}, FN={fn_def} | "
          f"Acc={accuracy_def:.3f} Sens={sensitivity_def:.3f} Spec={specificity_def:.3f}")
    print(f"Seuil opt.:  TP={tp_opt}, TN={tn_opt}, FP={fp_opt}, FN={fn_opt} | "
          f"Acc={accuracy_opt:.3f} Sens={sensitivity_opt:.3f} Spec={specificity_opt:.3f}")

    # --- Visualisation comparative ---
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    sns.heatmap(
        conf_matrix_default,
        annot=True, fmt='d',
        cmap='Blues',
        cbar=True, square=True,
        xticklabels=['Non-Conforme', 'Conforme'],
        yticklabels=['Non-Conforme', 'Conforme'],
        ax=axes[0]
    )
    axes[0].set_title(
        f'Seuil par défaut (0.5)\n'
        f'Acc={accuracy_def:.3f} | Sens={sensitivity_def:.3f} | Spec={specificity_def:.3f}\n'
        f'TP={tp_def}, TN={tn_def}, FP={fp_def}, FN={fn_def}',
        fontsize=12, fontweight='bold', pad=15
    )
    axes[0].set_ylabel('Vraie Classe', fontsize=11, fontweight='bold')
    axes[0].set_xlabel('Classe Prédite', fontsize=11, fontweight='bold')

    sns.heatmap(
        conf_matrix_optimal,
        annot=True, fmt='d',
        cmap='Greens',
        cbar=True, square=True,
        xticklabels=['Non-Conforme', 'Conforme'],
        yticklabels=['Non-Conforme', 'Conforme'],
        ax=axes[1]
    )
    axes[1].set_title(
        f'Seuil optimal ({optimal_threshold:.3f})\n'
        f'Acc={accuracy_opt:.3f} | Sens={sensitivity_opt:.3f} | Spec={specificity_opt:.3f}\n'
        f'TP={tp_opt}, TN={tn_opt}, FP={fp_opt}, FN={fn_opt}',
        fontsize=12, fontweight='bold', pad=15
    )
    axes[1].set_ylabel('Vraie Classe', fontsize=11, fontweight='bold')
    axes[1].set_xlabel('Classe Prédite', fontsize=11, fontweight='bold')

    fig.suptitle(
        f'Comparaison Seuil par Défaut vs Optimal - {best_model_name}',
        fontsize=15, fontweight='bold', y=1.02
    )

    plt.tight_layout()
    plt.show()